# 🎧 Projet — Telco Customer 360
## Plateforme client intelligente : churn, réclamations et rétention sur Azure

---

### 🎯 Contexte

Vous êtes Data Scientist dans l'équipe **Expérience Client** d'un opérateur télécom. Chaque mois, des milliers de clients résilient leur abonnement (**churn**), souvent après avoir montré des signaux d'insatisfaction : réclamations, dégradation de l'usage, facture jugée trop élevée. Retenir un client existant coûte **5 à 7 fois moins cher** que d'en acquérir un nouveau — la rétention est donc un enjeu business majeur.

Votre direction vous confie la construction d'un **POC de plateforme "Customer 360"**, hébergée sur **Microsoft Azure**, qui doit répondre à trois questions :

1. **Qui va partir ?** → un modèle prédictif de churn
2. **Pourquoi ?** → l'explicabilité du modèle + l'analyse des réclamations clients (texte libre)
3. **Que proposer pour le retenir ?** → un moteur de recommandation d'actions de rétention, sous forme de fiches destinées aux conseillers clientèle

### 📦 Livrables attendus

| # | Livrable | Format |
|---|---|---|
| 1 | Notebook d'analyse et de modélisation | `.ipynb` |
| 2 | Modèle de churn entraîné + rapport de performance | `.joblib` + métriques |
| 3 | "Liste rouge" des clients prioritaires | `.csv` |
| 4 | Fiches de rétention générées automatiquement | texte / JSON |
| 5 | Dashboard interactif déployé sur Azure | App Service (Streamlit) |
| 6 | Schéma d'architecture Azure du projet | image / diagramme |

### 🗂️ Données

- **Dataset principal** : [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — 7 043 clients, 21 variables : profil (ancienneté, type de contrat, mode de paiement), services souscrits (internet, streaming, sécurité...), facturation (mensuelle, totale) et la cible `Churn` (Yes/No).
- **Données à générer vous-même** : des tickets de réclamation en texte libre (voir Étape 5) — c'est ce qui rend le projet réaliste, car en entreprise les données ne sont jamais que tabulaires.

### 🧰 Stack technique

**Local** : Python, pandas, scikit-learn, XGBoost, SHAP, Streamlit.
**Azure** (application pratique de l'AZ-900) : Blob Storage, Azure Functions, Azure AI Language ou Azure OpenAI, App Service, Microsoft Entra ID, Cost Management.

💡 Créez un compte **Azure for Students** (100$ de crédit, sans carte bancaire, via votre email universitaire) ou un compte d'essai gratuit.

---
# Étape 1 — Mise en place de l'environnement Azure

**Objectif pédagogique** : manipuler concrètement les notions d'abonnement, de groupe de ressources et de gouvernance des coûts (domaine 1 et 3 de l'AZ-900).

**Ce que vous devez faire :**

1. Créer votre compte Azure (Students ou essai gratuit).
2. Créer un **groupe de ressources** `rg-telco360` dans la région `France Central`. Comprenez pourquoi on regroupe : un groupe de ressources se supprime d'un bloc, se facture d'un bloc, se gouverne d'un bloc.
3. Configurer un **budget** avec alerte à 50% et 80% dans **Cost Management**. C'est un réflexe professionnel : en entreprise, une VM oubliée = des centaines d'euros.
4. Explorer le portail : trouvez où se situent les régions, les zones de disponibilité, et ce que contient votre abonnement.

**Questions à vous poser (et à savoir répondre en entretien) :**
- Quelle est la différence entre un abonnement, un groupe de gestion et un groupe de ressources ?
- Pourquoi choisir France Central plutôt que East US pour un opérateur français ? (latence, souveraineté des données, RGPD)

✅ **Critère de réussite** : capture d'écran de votre groupe de ressources + budget configuré.

---
# Étape 2 — Ingestion des données dans Azure Blob Storage

**Objectif pédagogique** : comprendre le stockage objet, les tiers d'accès et la redondance (domaine 2 de l'AZ-900).

**Ce que vous devez faire :**

1. Créer un **compte de stockage** avec redondance **LRS** (Locally Redundant Storage). Demandez-vous : pourquoi LRS suffit ici, et dans quel cas une banque comme le Crédit Mutuel exigerait du GRS (géo-redondance) ?
2. Créer deux **conteneurs** : `raw-data` (données brutes) et `processed-data` (données nettoyées).
3. Téléverser le CSV Kaggle dans `raw-data` — d'abord via le portail, puis **en Python** avec le SDK `azure-storage-blob` (c'est la version qu'un recruteur veut voir).
4. Expérimenter les **tiers d'accès** : le fichier brut en tier *Hot*, puis configurez une **règle de lifecycle management** qui bascule automatiquement en *Cool* après 30 jours.

**Point sécurité** : ne mettez JAMAIS la chaîne de connexion en dur dans le code. Utilisez une variable d'environnement, et documentez que la bonne pratique en production est **Azure Key Vault**.

✅ **Critère de réussite** : lire le CSV directement depuis le Blob Storage dans votre notebook.

In [ ]:
# ✍️ À vous de jouer


---
# Étape 3 — Exploration et préparation des données (EDA)

**Objectif pédagogique** : comprendre les données avant de modéliser — l'étape que les débutants bâclent et que les seniors soignent.

**Ce que vous devez faire :**

1. **Audit qualité** : le piège classique de ce dataset est la colonne `TotalCharges`, stockée en texte avec des valeurs vides pour les nouveaux clients. Identifiez-le, corrigez-le, et expliquez pourquoi ces valeurs sont vides (indice : regardez `tenure`).
2. **Analyse de la cible** : quel est le taux de churn global ? Le dataset est-il déséquilibré ? Quelles conséquences pour la modélisation ?
3. **Analyse univariée et bivariée** — produisez au minimum :
   - Distribution de l'ancienneté (`tenure`) selon le churn
   - Facture mensuelle selon le churn
   - Taux de churn par type de contrat (`Contract`), par mode de paiement, par type d'internet
4. **Synthèse métier** : rédigez 4-5 phrases résumant le "portrait-robot" du client qui résilie. C'est cet exercice de traduction chiffres → business qui fait la différence en entretien.

**Résultat attendu** (à retrouver vous-même) : les clients en contrat mensuel, récents, avec la fibre et une facture élevée churnent massivement ; les clients engagés 2 ans presque jamais.

✅ **Critère de réussite** : 3+ visualisations propres et titrées + synthèse métier écrite.

In [ ]:
# ✍️ À vous de jouer


---
# Étape 4 — Modèle de churn (XGBoost) et explicabilité (SHAP)

**Objectif pédagogique** : construire un modèle performant ET explicable — en assurance comme en télécom, un score sans explication est inutilisable par le métier.

**Ce que vous devez faire :**

1. **Feature engineering** — créez au moins 3 variables métier, par exemple :
   - `nb_services` : nombre de services souscrits
   - `charge_par_service` : facture mensuelle / nombre de services (détecte le sentiment de "payer trop cher")
   - `nouveau_client` : ancienneté ≤ 6 mois
2. **Encodage** des variables catégorielles (justifiez votre choix : LabelEncoder vs One-Hot).
3. **Split** train/test stratifié (80/20). Pourquoi stratifié ? À cause du déséquilibre de la cible.
4. **Entraîner un XGBoost**. Gérez le déséquilibre avec `scale_pos_weight` — sachez expliquer ce que fait ce paramètre.
5. **Évaluer** : AUC, precision, recall, matrice de confusion. Question centrale à trancher **du point de vue métier** : vaut-il mieux maximiser la precision ou le recall ? (Indice : rater un client qui part coûte plus cher que d'appeler un client qui serait resté.)
6. **SHAP** :
   - Un `summary_plot` global : quelles variables pilotent le churn ?
   - Une fonction `expliquer_client(id)` qui retourne, pour un client donné, sa probabilité de churn et ses 3 principaux facteurs de risque.

✅ **Critère de réussite** : AUC ≥ 0.83 + fonction d'explication individuelle fonctionnelle.

In [ ]:
# ✍️ À vous de jouer


---
# Étape 5 — Génération des tickets de réclamation (données non structurées)

**Objectif pédagogique** : travailler avec du texte libre, comme dans un vrai CRM — et découvrir la génération de données synthétiques par LLM.

**Ce que vous devez faire :**

1. Générer entre 800 et 1500 **tickets de réclamation** en français, rattachés à des `customerID` existants, répartis sur 4 motifs : `réseau`, `facturation`, `résiliation`, `service_client`.
2. **Contrainte de réalisme** : les clients churners doivent avoir une probabilité de réclamation nettement plus élevée que les autres (ex. 40% vs 5%). Justifiez ce choix : dans la réalité, la réclamation est un signal avancé du churn.
3. Deux approches possibles (faites au moins la première) :
   - **Templates + aléatoire** : une banque de phrases par motif, tirage aléatoire.
   - **Azure OpenAI** : prompter un LLM pour générer des verbatims variés et réalistes ("génère 50 réclamations de clients mécontents du débit internet, ton familier, en français").
4. Stocker le résultat en CSV dans le conteneur `raw-data`.

**Réflexion à mener** : quels sont les biais et limites des données synthétiques ? Pourquoi un modèle entraîné dessus ne serait pas directement déployable en production ?

✅ **Critère de réussite** : un DataFrame de tickets avec `customerID`, `texte`, `motif_reel`.

In [ ]:
# ✍️ À vous de jouer


---
# Étape 6 — Analyse NLP des tickets : classification des motifs et sentiment

**Objectif pédagogique** : transformer du texte en signal exploitable, et comparer une approche artisanale à un service managé Azure.

**Ce que vous devez faire :**

1. **Baseline locale** : un classifieur de motifs simple (mots-clés ou TF-IDF + régression logistique). Mesurez son accuracy contre `motif_reel`.
2. **Analyse de sentiment** : distinguez au minimum "négatif" et "très négatif" (les verbatims avec menace de résiliation ou vocabulaire fort).
3. **Version Azure managée** : créez une ressource **Azure AI Language** (tier gratuit F0 : 5000 requêtes/mois) et passez vos tickets dans son API d'analyse de sentiment. Comparez avec votre baseline.
4. **Question d'architecte** : dans quels cas préférer un service managé (AI Language) vs son propre modèle ? Pensez coût, maintenance, RGPD, personnalisation — c'est une question type AZ-900 (SaaS vs build).

✅ **Critère de réussite** : chaque ticket enrichi avec `motif_predit` et `sentiment` + un paragraphe comparant les deux approches.

In [ ]:
# ✍️ À vous de jouer


---
# Étape 7 — La "liste rouge" : croisement churn × réclamations

**Objectif pédagogique** : c'est ici que le projet devient un produit. On croise deux signaux pour prioriser l'action.

**Ce que vous devez faire :**

1. Scorer **toute la base clients** avec votre modèle de churn.
2. Joindre les tickets de réclamation (attention à la cardinalité : un client peut avoir plusieurs tickets — comment agréger ?).
3. Construire la **liste rouge** : clients avec `proba_churn ≥ 0.6` **ET** au moins une réclamation. Triée par risque décroissant.
4. Enrichir chaque ligne : ancienneté, contrat, facture, motif de réclamation, sentiment, top 3 facteurs SHAP.
5. Exporter en CSV dans le conteneur `processed-data`.

**Question métier** : pourquoi 0.6 et pas 0.5 ? Réfléchissez en termes de **capacité de traitement** : si l'équipe rétention ne peut appeler que 50 clients/semaine, le seuil doit s'ajuster à cette contrainte, pas à une convention statistique.

✅ **Critère de réussite** : un CSV exploitable tel quel par une équipe métier.

In [ ]:
# ✍️ À vous de jouer


---
# Étape 8 — Moteur de rétention : fiches conseiller générées par LLM

**Objectif pédagogique** : la partie "hype" — combiner ML prédictif et IA générative dans un cas d'usage concret.

**Ce que vous devez faire :**

1. Définir une **matrice d'actions de rétention** : pour chaque motif de réclamation, une action type (ex. facturation → audit de facture + remboursement des options non souscrites ; réseau → diagnostic prioritaire + geste commercial...).
2. Créer une fonction `fiche_conseiller(client)` qui produit une fiche structurée : identité, risque de churn, facteurs SHAP, verbatim de la réclamation, action recommandée.
3. **Version LLM (Azure OpenAI)** : déployer un modèle léger (ex. gpt-4o-mini) et prompter pour générer une fiche **rédigée** en langage naturel. Travaillez le prompt : rôle ("tu es un assistant pour conseillers clientèle télécom"), contexte injecté (données du client), format de sortie imposé.
4. **Garde-fous** : que se passe-t-il si le LLM invente une remise que l'entreprise ne propose pas ? Contraignez la sortie (liste d'actions autorisées dans le prompt) et documentez cette réflexion — c'est exactement la problématique des LLM en entreprise.

✅ **Critère de réussite** : 10 fiches générées pour le top 10 de la liste rouge, sans hallucination d'offre.

In [ ]:
# ✍️ À vous de jouer


---
# Étape 9 — Dashboard et déploiement sur Azure App Service

**Objectif pédagogique** : passer du notebook au produit déployé — la compétence qui distingue un candidat.

**Ce que vous devez faire :**

1. Développer une app **Streamlit** avec 3 vues :
   - **Vue portefeuille** : KPIs globaux (taux de churn prédit, nb de clients en liste rouge, répartition des motifs de réclamation)
   - **Vue client** : recherche par `customerID` → score, facteurs SHAP, historique des tickets, fiche de rétention
   - **Vue liste rouge** : tableau triable/filtrable, export CSV
2. L'app doit lire ses données **depuis le Blob Storage** (pas de CSV local) — c'est ce qui en fait une vraie app cloud.
3. **Déployer sur Azure App Service** (plan gratuit F1 ou Basic B1). Vous découvrirez le concept de plan App Service, le déploiement via Git ou GitHub Actions.
4. **Sécuriser** : activer l'authentification **Microsoft Entra ID** sur l'App Service (fonctionnalité "Easy Auth", quelques clics) pour que seuls les comptes autorisés accèdent au dashboard.
5. **Automatisation bonus** : une **Azure Function** avec Blob Trigger qui relance le scoring dès qu'un nouveau fichier client arrive dans `raw-data`.

✅ **Critère de réussite** : une URL publique (protégée par login) que vous pouvez montrer en entretien depuis votre téléphone.

In [ ]:
# ✍️ À vous de jouer


---
# Étape 10 — Documentation, coûts et pitch

**Objectif pédagogique** : un projet non documenté n'existe pas. Et savoir parler coûts = crédibilité immédiate.

**Ce que vous devez faire :**

1. **Schéma d'architecture** : dessinez le flux complet (draw.io ou l'outil de diagramme d'Azure). Blob → Function → Modèle → AI Language/OpenAI → App Service, avec Entra ID et Cost Management en transverse.
2. **Bilan des coûts** : relevez dans Cost Management ce que le projet a réellement consommé, service par service. Estimez le coût mensuel si l'app tournait en continu pour 100 000 clients.
3. **README GitHub** soigné : contexte, architecture, résultats (AUC, exemples de fiches), instructions de reproduction.
4. **Pitch de 2 minutes** (écrivez-le) : *"J'ai construit une plateforme Customer 360 sur Azure qui identifie les clients à risque de churn (XGBoost, AUC 0.84), analyse leurs réclamations en NLP, et génère des fiches de rétention actionnables via Azure OpenAI, le tout déployé sur App Service avec authentification Entra ID."*

### 🧭 Correspondance avec le programme AZ-900

| Domaine AZ-900 | Où vous l'avez pratiqué |
|---|---|
| Concepts du cloud (IaaS/PaaS/SaaS, responsabilité partagée) | Function (serverless), App Service (PaaS), AI Language (SaaS) |
| Architecture Azure (régions, groupes de ressources, abonnements) | Étape 1 |
| Stockage (tiers, redondance, lifecycle) | Étape 2 |
| Identité et sécurité (Entra ID, Key Vault) | Étapes 2 et 9 |
| Gestion et gouvernance (Cost Management, budgets, Monitor) | Étapes 1 et 10 |

### ⏱️ Planning indicatif (5-6 semaines à temps partiel)

- **Semaine 1** : Étapes 1-2 (Azure + stockage)
- **Semaine 2** : Étapes 3-4 (EDA + modèle)
- **Semaine 3** : Étapes 5-6 (tickets + NLP)
- **Semaine 4** : Étapes 7-8 (liste rouge + LLM)
- **Semaines 5-6** : Étapes 9-10 (déploiement + doc)

Bon courage 🚀